## Preparing the data for the random forest

In [14]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
#from imblearn.pipeline import make_pipeline
#from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import accuracy_score, precision_recall_curve, precision_recall_fscore_support, f1_score, precision_score, recall_score, roc_auc_score, roc_curve, auc
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from joblib import dump
import numpy as np
import matplotlib.pyplot as plt
#from imblearn.under_sampling import RandomUnderSampler
import os
from osgeo import gdal
import time
import functools
import seaborn as sns
import matplotlib.ticker as ticker

### Read initial csv data file

In [15]:
def parse_data(file_path):
    df = pd.read_csv(file_path, delimiter=';', index_col=False)
    #print("Original dataframe", df.head())
    print("Original dataframe shape", df.shape)
    #print all column names
    print("Column names", df.columns)
    # print all the nan values in the column nbr
    print("Nan values in the column nbr", df['NBR'].isna().sum())
    # Count unique numerical IDs where 'NBR' is NaN
    if 'numerical_id' in df.columns:
        unique_ids_with_nan = df.loc[df['NBR'].isna(), 'numerical_id'].nunique()
        print("Unique numerical IDs with NaN values in 'NBR':", unique_ids_with_nan)
    else:
        print("Column 'numerical_id' not found in the dataset!")
    print("Percentage of NaN values in 'NBR':", df['NBR'].isna().sum()/df.shape[0])
    print("Number of unique numerical IDs:", df['numerical_id'].nunique())
    print("percentage of unique numerical IDs with nan values in 'NBR':", unique_ids_with_nan/df['numerical_id'].nunique())
    return df

#file_path = "/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/Data/Training/clear_forest_dataset_timesync_alltiles_landsatbands_indices3_classes_ed4_calibration.csv"
#file_path = '/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/Data/Training/forest_dataset_timesync_alltiles_landsatbands_indices3_classes_ed2_calibration.csv'
file_path = "/home/ubuntu/work/saved_data/landsat_disturbance_detection/1D_U_Net/Data/Training/forest_dataset_timesync_alltiles_landsatbands_indices3_classes_ed5_calibration.csv"
df = parse_data(file_path)

Original dataframe shape (567084, 25)
Column names Index(['fid', 'country', 'plotid', 'year', 'class_level1', 'class_level2',
       'new_class2_v5', 'uniqueid', 'numerical_id', 'x', 'y', 'Tile_ID',
       'merged_id', 'BLU', 'GRN', 'RED', 'NIR', 'SW1', 'SW2', 'NBR', 'NDVI',
       'TCB', 'TCG', 'TCW', 'DIn'],
      dtype='object')
Nan values in the column nbr 37717
Unique numerical IDs with NaN values in 'NBR': 10690
Percentage of NaN values in 'NBR': 0.06651042879009106
Number of unique numerical IDs: 17069
percentage of unique numerical IDs with nan values in 'NBR': 0.6262815630675493


### Encode the disturbances

In [16]:
def encoding_disturbances(df):
    #df['class'] = df['new_class2']
    df["class"] = df["new_class2_v5"]
    #np.where(df['class_level2'] == 'stand-replacing disturbance', 1, 0)

    return df

df = encoding_disturbances(df)
print("Shape after encoding disturbances", df.shape)
#print(df.head)

Shape after encoding disturbances (567084, 26)


### Remove all unnecessary columns

In [17]:
def drop_columns(df):
    df = df[['year', 'numerical_id', 'class', 'NDVI', 'NBR', 'TCB', 'TCG', 'TCW', 'DIn']]
    return df

df = drop_columns(df)
print("Shape after dropping columns", df.shape)
print(df.head)

Shape after dropping columns (567084, 9)
<bound method NDFrame.head of         year  numerical_id  class      NDVI       NBR     TCB     TCG    TCW  \
0       1985             1      0  0.903261  0.691500  3964.0  2999.0 -396.0   
1       1986             1      0  0.846658  0.697250  3981.0  2713.0 -168.0   
2       1987             1      0  0.884108  0.714342  3884.0  2926.0 -127.0   
3       1988             1      0  0.862136  0.678677  4514.0  3146.0 -284.0   
4       1989             1      0  0.855061  0.706518  4341.0  2956.0 -218.0   
...      ...           ...    ...       ...       ...     ...     ...    ...   
567079  2014         19818      0  0.845379  0.790024  2165.0  1591.0  381.0   
567080  2015         19818      0  0.789429  0.693477  1968.0  1254.0  132.0   
567081  2016         19818      0       NaN       NaN     NaN     NaN    NaN   
567082  2017         19818      0       NaN       NaN     NaN     NaN    NaN   
567083  2018         19818      0  0.825513  0.72

### Remove all time series with less than 30 years

In [18]:
def count_numerical_ids(df):
    id_counts = df['numerical_id'].value_counts()

    # Count occurrences of each numerical_id
    id_counts = df['numerical_id'].value_counts()
    # Filter numerical_id's that appear less than 34 times
    num_ids_less_than_34 = (id_counts < 34).sum()

    print(f"Number of numerical_id's that appear in less than 34 years: {num_ids_less_than_34}")
    return id_counts

In [19]:
def filter_numerical_ids(df, min_count=30):
    """
    Remove numerical_id entries that appear less than min_count times.
    """
    id_counts = df['numerical_id'].value_counts()
    
    # Get the IDs that meet the minimum count requirement
    valid_ids = id_counts[id_counts >= min_count].index

    # Filter the DataFrame
    filtered_df = df[df['numerical_id'].isin(valid_ids)]
    
    return filtered_df

# Apply filtering
df_filtered = filter_numerical_ids(df, min_count=30)

# Print the count after filtering
print(count_numerical_ids(df_filtered))

Number of numerical_id's that appear in less than 34 years: 8925
numerical_id
1        34
10796    34
10733    34
10734    34
10438    34
         ..
445      30
754      30
755      30
1336     30
19209    30
Name: count, Length: 17048, dtype: int64


### Apply linear interpolation

In [20]:
df_linear = df_filtered.interpolate()

### Compute the differences between years for NDVI and NBR each: sort for year afterwards

In [21]:
df_sorted = df_linear.sort_values(by=["numerical_id", "year"])
df_sorted.head(10)

,year,numerical_id,class,NDVI,NBR,TCB,TCG,TCW,DIn
0,1985,1,0,0.903261,0.691500,3964.0,2999.0,-396.0,-3.122869
1,1986,1,0,0.846658,0.697250,3981.0,2713.0,-168.0,-2.701317
2,1987,1,0,0.884108,0.714342,3884.0,2926.0,-127.0,-3.254909
3,1988,1,0,0.862136,0.678677,4514.0,3146.0,-284.0,-3.075986
4,1989,1,0,0.855061,0.706518,4341.0,2956.0,-218.0,-2.374364
5,1990,1,0,0.808742,0.670267,4332.0,2699.0,-197.0,-2.698040
6,1991,1,0,0.893613,0.683797,4682.0,3434.0,-278.0,-2.840667
7,1992,1,0,0.882402,0.719419,4012.0,3056.0,-183.0,-3.155146
8,1993,1,0,0.895533,0.703258,4907.0,3728.0,-209.0,-3.339898
9,1994,1,0,0.845774,0.699545,4516.0,3108.0,-101.0,-3.509480


In [22]:
# Shift all columns within the same numerical_id
df_sorted["NDVI_diff"] = df_sorted["NDVI"] - df_sorted.shift(1)["NDVI"]
df_sorted["NBR_diff"] = df_sorted["NBR"] - df_sorted.shift(1)["NBR"]
df_sorted["TCB_diff"] = df_sorted["TCB"] - df_sorted.shift(1)["TCB"]
df_sorted["TCG_diff"] = df_sorted["TCG"] - df_sorted.shift(1)["TCG"]
df_sorted["TCW_diff"] = df_sorted["TCW"] - df_sorted.shift(1)["TCW"]
df_sorted["DIn_diff"] = df_sorted["DIn"] - df_sorted.shift(1)["DIn"]

# Set NaN where numerical_id changes (to prevent incorrect subtraction)
df_sorted.loc[df_sorted["numerical_id"] != df_sorted["numerical_id"].shift(1), ["NDVI_diff", "NBR_diff", "TCB_diff", "TCG_diff", "TCW_diff", "DIn_diff"]] = None

In [23]:
nan_details = df_sorted.isna().groupby(df_sorted["year"]).sum()

In [24]:
# Find the first recorded year for each numerical_id
first_years = df_sorted.groupby("numerical_id")["year"].min()

# Mark rows that are the first year for each numerical_id
df_sorted.loc[:, "is_first_year"] = df_sorted["year"] == df_sorted["numerical_id"].map(first_years)

# Find all rows with NaN values
nan_rows = df_sorted.isna().any(axis=1)

# Check if ALL NaN rows are in the first recorded year
all_nans_in_first_year = df_sorted.loc[nan_rows, "is_first_year"].all()

# Print the result
print("Are all NaN values in the first recorded year for each numerical_id?", all_nans_in_first_year)
df_sorted = df_sorted.drop(columns=["is_first_year"])

Are all NaN values in the first recorded year for each numerical_id? True


In [25]:
df_sorted = df_sorted.dropna()
print(df_sorted.head(10))

    year  numerical_id  class      NDVI       NBR     TCB     TCG    TCW  \
1   1986             1      0  0.846658  0.697250  3981.0  2713.0 -168.0   
2   1987             1      0  0.884108  0.714342  3884.0  2926.0 -127.0   
3   1988             1      0  0.862136  0.678677  4514.0  3146.0 -284.0   
4   1989             1      0  0.855061  0.706518  4341.0  2956.0 -218.0   
5   1990             1      0  0.808742  0.670267  4332.0  2699.0 -197.0   
6   1991             1      0  0.893613  0.683797  4682.0  3434.0 -278.0   
7   1992             1      0  0.882402  0.719419  4012.0  3056.0 -183.0   
8   1993             1      0  0.895533  0.703258  4907.0  3728.0 -209.0   
9   1994             1      0  0.845774  0.699545  4516.0  3108.0 -101.0   
10  1995             1      0  0.876190  0.754110  4479.0  3391.0  115.0   

         DIn  NDVI_diff  NBR_diff  TCB_diff  TCG_diff  TCW_diff  DIn_diff  
1  -2.701317  -0.056603  0.005750      17.0    -286.0     228.0  0.421553  
2  -3.25490

### Export to csv

In [26]:
df_sorted.to_csv("Data_for_RF.csv", index=False)